In [3]:
import folium
from folium.plugins import AntPath
import pandas as pd
import geopandas as gpd
import json
from gerar_mapa import BASE, LINK_HOME

print(LINK_HOME)

https://rogerfidelis.github.io/ofi/index.html


In [ ]:
import folium
from folium.plugins import AntPath
import pandas as pd
import geopandas as gpd
import json
from gerar_mapa import BASE, LINK_HOME

# =========================
# MAPA BASE
# =========================
EMPRESAS = ["cbo", "bram", "starnav"]

for EMPRESA in EMPRESAS:
    
    DATA_INICIO = "2026-07-16"
    DATA_FIM = "2026-09-13"

    DATA_INICIO = pd.to_datetime(DATA_INICIO)
    DATA_FIM = pd.to_datetime(DATA_FIM) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    

    SURVEY_STARNAV = BASE / "dados" / f"SURVEY_{EMPRESA}.xlsx"
    SURVEY_POSICOES = BASE / "dados" / "SURVEY_POSICOES.xlsx"

    dfs = pd.read_excel(SURVEY_STARNAV, sheet_name=None)
    dfs_2 = pd.read_excel(SURVEY_POSICOES)

    vessels = pd.read_excel(BASE / "dados" / "SURVEY_VESSELS"/"SURVEY_VESSELS.xlsx")

    for celula in vessels[EMPRESA].dropna():
        #print (celula)
        mapa = folium.Map(
            location=[-22.872174, -41.983981],
            zoom_start=9
        )

        # =========================
        # SHAPEFILES (ANP)
        # =========================
        shp_campos = gpd.read_file(
        BASE /
        "shapefiles" /
        "CAMPOS_PRODUCAO_SIRGAS" /
        "CAMPOS_PRODUCAO_SIRGASPolygon.shp"
    )

        if shp_campos.crs is None or shp_campos.crs.to_epsg() != 4326:
            shp_campos = shp_campos.to_crs(epsg=4326)

        folium.GeoJson(
            data=json.loads(shp_campos.to_json()),
            name="Campos de Produção (ANP)",
            style_function=lambda x: {
                "fillColor": "yellow",
                "color": "orange",
                "weight": 2,
                "fillOpacity": 0.3
            },
            tooltip=folium.GeoJsonTooltip(
                fields=["NOM_CAMPO"],
                aliases=["Campo:"]
            )
        ).add_to(mapa)

        
        pontos = dfs_2.dropna(subset=["latitude", "longitude"])

        cores = {
                "porto":       "red",
                "navio sonda": "blue",
                "FPSO":        "purple",
                "Semi-Sub/Prod/Perfuração": "green",
                "Semi-Sub/Perfuração":      "orange",
                "Fixa (Habitada)":          "darkred",
                "Semi-Sub/Produção":        "cadetblue",
                "estaleiro": "white"
                }

        # =========================
        # PONTOS FIXOS + ZONA 500 m
        # =========================
        for _, p in pontos.iterrows():
            folium.CircleMarker(
                [p["latitude"], p["longitude"]],
                 radius=6,
                color=cores[p["tipo"]],
                fill=True,
                fill_opacity=0.7,
                popup=f'{p["unidade"]} - {p["tipo"]}'
            ).add_to(mapa)



        
        cbo = dfs[celula.upper()].copy()

        # Converte a data da posição para datetime
        cbo["data_reportada"] = pd.to_datetime(
            cbo["data_reportada"],
            dayfirst=True,
            errors="coerce"
        )

        # Filtra o período desejado
        cbo = cbo[
            (cbo["data_reportada"] >= DATA_INICIO) &
            (cbo["data_reportada"] <= DATA_FIM)
        ].copy()

        # Remove posições inválidas
        cbo = cbo.dropna(subset=["lat_a", "lon_a"])

        # Se não houver dados no período, não gera o mapa
        if cbo.empty:
            print(
                f"{celula}: nenhuma posição encontrada entre "
                f"{DATA_INICIO.date()} e {DATA_FIM.date()}"
            )
            continue

        trajetoria = cbo[["lat_a", "lon_a"]].values.tolist()
        print(trajetoria)

        
        # =========================
        # POSIÇÕES DA EMBARCAÇÃO
        # =========================
        
        for _, p in cbo.iterrows():
            folium.CircleMarker(
                [p["lat_a"], p["lon_a"]],
                radius=3,
                color="blue",
                fill=True,
                popup=f"""
                <b>Hora:</b> {p["hora_reportada"]}<br>
                <b>Data:</b> {p["data_reportada"]}
                <b>Status:</b> {p["status"]}
                """
             ).add_to(mapa)

        # =========================
        # TRAJETÓRIA
        # =========================

        AntPath(
            locations=trajetoria,
            color="blue",
            weight=5,
            delay=900,
            dash_array=[5, 10],
            pulse_color="white"
        ).add_to(mapa)

        # =========================
        # INÍCIO E FIM
        # =========================
        inicio = cbo.iloc[0]
        fim = cbo.iloc[-1]

        folium.Marker(
            trajetoria[0],
            icon=folium.Icon(color="green", icon="play"),
            popup=f'INÍCIO: {inicio["hora_reportada"]} - {inicio["data_reportada"]}'
        ).add_to(mapa)

        folium.Marker(
            trajetoria[-1],
            icon=folium.Icon(color="red", icon="stop"),
            popup=f'FIM: {fim["hora_reportada"]} - {fim["data_reportada"]}'
        ).add_to(mapa)

        # =========================
        # SALVAR
        # =========================
        #mapa.save("cbo_copacabana.html")
        #mapa.save("bram_bravo.html")
        #mapa.save("starnav_libra.html")
        ##########################################################################

        legenda = """
        <div style="
        position:fixed;
        bottom:25px;
        right:25px;

        width:min(280px, 90vw);
        @media (max-width:768px){

            .legenda{

                bottom:15px;

                right:15px;

                left:15px;

                width:auto;

                font-size:12px;

                padding:10px;

            }

        }

        background:#0B1F3A;
        border:2px solid #1CA3EC;
        border-radius:10px;
        padding:15px;
        color:white;
        font-family:Arial;
        font-size:14px;
        z-index:9999;
        ">

        <h3 style="margin-top:0;color:#1CA3EC;">
        Legenda
        </h3>

        ● <span style="color:red;">Porto</span><br>
        ● <span style="color:purple;">FPSO</span><br>
        ● <span style="color:blue;">Navio Sonda</span><br>
        ● <span style="color:green;">Semi-Sub/Prod/Perfuração</span><br>
        ● <span style="color:yellow;">Semi-Sub/Perfuração</span><br>
        ● <span style="color:darkred;">Plataforma Fixa</span><br>
        ● <span style="color:white;">Estaleiro</span><br><br>



        <span style="color:#00BFFF;">━ ━ ━ ━ </span> Trajetória AIS<br>

        🟢 Início<br>

        🔴 Última posição

        </div>
        """

        mapa.get_root().html.add_child(folium.Element(legenda))

      #######################################################################


        mapa.save(BASE/"mapas"/EMPRESA/f'{celula.replace(" ", "_")}.html')
        #print(celula.replace(" ", "_") + ".html")
            

C:\Users\roger\Anaconda3\envs\geo_video\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


[[-23.88, -43.12], [-22.86, -43.12], [-22.86, -43.12], [-22.864996, -43.129906], [-22.86491, -43.12981], [-22.864872, -43.129959], [-22.864862, -43.129883], [-22.864923, -43.129906], [-22.865023, -43.129768], [-22.865009, -43.129772], [-22.865034, -43.129921], [-22.865017, -43.129894], [-22.864929, -43.12989], [-22.864916, -43.12994], [-22.864958, -43.129799], [-22.866549, -43.129745], [-22.86651, -43.12978], [-22.86651, -43.12978], [-22.86651, -43.12978]]
[[-24.78, -42.75], [-24.78, -42.75], [-24.91, -42.78], [-22.895718, -43.205105], [-22.83861, -43.140579], [-22.893656, -43.212616], [-22.838964, -43.141293], [-22.838753, -43.140545], [-22.893532, -43.212624], [-23.503202, -42.987629], [-23.503202, -42.987629], [-23.503202, -42.987629], [-24.771143, -42.724731], [-24.771143, -42.724731], [-25.602434, -42.822323], [-25.602636, -42.822712], [-25.623302, -42.839912], [-25.596003, -42.853485], [-25.608391, -42.843231]]
[[-22.87, -43.12], [-22.84, -43.12], [-22.84, -43.12], [-22.842417, -

[[-22.85, -43.14], [-22.85, -43.14], [-22.85, -43.14], [-22.853, -43.1416], [-22.854073, -43.141575], [-22.854708, -43.142048], [-22.854305, -43.14172], [-22.85441, -43.140862], [-22.854092, -43.140614], [-22.853275, -43.140945], [-22.853411, -43.140648], [-22.853275, -43.140881], [-22.854031, -43.141052], [-22.843912, -43.136387], [-22.844355, -43.136986], [-22.844244, -43.136971], [-22.844484, -43.136787], [-22.845036, -43.136059], [-22.845169, -43.135925]]
[[-22.88, -43.13], [-23.81, -43.77], [-24.7034, -42.409164], [-24.771299, -42.018955], [-24.700127, -42.407646], [-24.703123, -42.408306], [-24.70385, -42.405991], [-24.703659, -42.405407], [-24.703659, -42.405407], [-24.703659, -42.405407], [-24.703659, -42.405407], [-24.703659, -42.405407], [-24.705826, -42.404228], [-24.510635, -42.499897], [-22.847673, -43.138664], [-22.847204, -43.138699], [-22.847364, -43.138481]]
[[-23.0, -40.65], [-22.946316, -40.703674], [-23.056684, -40.697968], [-22.940487, -40.710594], [-22.946106, -40

[[-22.84, -43.13], [-24.31, -42.43], [-22.438591, -41.485249], [-21.855236, -41.009159], [-21.855236, -41.009174], [-22.135359, -40.911648], [-23.478909, -42.063766], [-24.508404, -42.528587], [-24.508404, -42.528587], [-24.534618, -42.202091], [-24.534618, -42.202091], [-24.534618, -42.202091], [-22.84856, -43.146961], [-23.2171, -43.022026], [-24.050817, -42.173805], [-24.050817, -42.173805], [-21.855045, -41.008755], [-21.855047, -41.00877]]
[[-21.88, -41.01], [-21.924438, -40.174892], [-21.866444, -41.015366], [-21.853306, -40.948029], [-21.853842, -40.948315], [-22.156605, -40.953365], [-22.376347, -41.735321], [-22.376244, -41.735363], [-22.375751, -41.733986], [-22.375643, -41.733849], [-22.375423, -41.733814], [-22.37529, -41.734035], [-21.990736, -39.760494], [-21.979481, -39.764145], [-21.944145, -39.714108], [-21.968632, -39.826981], [-21.973223, -40.256977], [-21.865593, -41.015862]]
[[-21.86, -41.01], [-22.717539, -40.690468], [-22.708782, -40.692547], [-23.137703, -42.543